# FireWorkers: learned gated communication on a JAX wildfire task (E1-E9)

**Notebook settings (right panel):** Accelerator = **GPU T4 x2**, Internet = **on**.
Run with **Save Version -> Save & Run All (Commit)** so it keeps running after you close the tab.

One session runs as much of the plan as fits in `BUDGET_HOURS`, then saves and stops cleanly.
To continue next session: open this notebook, **Add Input -> Notebook Output -> (this notebook's last version)**, and commit again.
Cell 3 restores `runs/` and `results/` from that input; finished phases are skipped and an interrupted
training batch resumes from its last checkpoint. See `kaggle/README_KAGGLE.md` for the full guide.

In [ ]:
# ---- 1. settings
BUDGET_HOURS = 11.0          # Kaggle GPU sessions stop at 12 h; leave time for evaluation + packaging
PHASES = None                # None = whole plan in order, or e.g. ["e1", "e2_pilot", "e2"]
UPDATES = None               # training updates per run (None = configs/algo/mappo.yaml); see the benchmark cell
SEEDS = [0, 1, 2, 3, 4]
GIT_URL = ""                 # optional: your GitHub copy of the project (used if no Kaggle dataset has it)
RUN_TESTS = True
RUN_BENCHMARK = True

In [ ]:
# ---- 2. get the code: a Kaggle dataset containing the project, or git clone
import os, glob, shutil, subprocess, sys
WORK = "/kaggle/working/wildfire-comm-marl"
if not os.path.exists(WORK):
    cand = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/scripts/run_plan.py", recursive=True)]
    cand = [os.path.dirname(c) for c in cand]
    # prefer a code dataset over a previous session's output (both contain the code)
    cand.sort(key=lambda c: os.path.exists(os.path.join(c, "results", "plan_status.json")))
    if cand:
        shutil.copytree(cand[0], WORK, ignore=shutil.ignore_patterns("runs", "results"))
        print("code copied from", cand[0])
    elif GIT_URL:
        subprocess.check_call(["git", "clone", "--depth", "1", GIT_URL, WORK])
    else:
        raise SystemExit("Add the project as a Kaggle dataset (zip of wildfire-comm-marl) or set GIT_URL")
os.chdir(WORK)
print(os.getcwd())

In [ ]:
# ---- 3. restore progress from a previous session (notebook output attached as input)
import json, pathlib
prev = [p for p in glob.glob("/kaggle/input/**/results/plan_status.json", recursive=True)]
if prev:
    src = pathlib.Path(prev[0]).parents[1]
    for sub in ("runs", "results"):
        if (src / sub).exists():
            shutil.copytree(src / sub, pathlib.Path(WORK) / sub, dirs_exist_ok=True)
    print("restored from", src, "->", json.loads((pathlib.Path(WORK) / "results/plan_status.json").read_text()))
else:
    print("no previous session found: starting fresh")

In [ ]:
# ---- 4. install pinned JAX (CUDA 12 wheels bundle CUDA/cuDNN; T4 = sm_75) and the rest
!pip install -q -U "jax[cuda12]==0.6.2" flax==0.10.7 optax==0.2.8 hydra-core==1.3.8 omegaconf==2.3.1 rliable==1.2.0 2>&1 | tail -2
import jax
print(jax.__version__, jax.devices())
assert jax.device_count() >= 1 and jax.devices()[0].platform == "gpu", "no GPU: set Accelerator = GPU T4 x2"
if jax.device_count() < 2:
    print("WARNING: only one GPU visible; runs will use one device")

In [ ]:
# ---- 5. fast correctness checks (env port, comm, analysis); ~2-4 min
if RUN_TESTS:
    !python -m pytest tests/test_wildfire.py tests/test_comm.py tests/test_analysis.py -q -p no:cacheprovider 2>&1 | tail -3

In [ ]:
# ---- 6. throughput on THIS session's GPUs and projected hours for the remaining plan
if RUN_BENCHMARK:
    !python scripts/benchmark.py --batches 1 8 24

In [ ]:
# ---- 7. run the plan (stops cleanly before BUDGET_HOURS; exit code 3 = budget reached, resume next session)
args = ["--budget-hours", str(BUDGET_HOURS), "--seeds", *map(str, SEEDS)]
if PHASES: args += ["--phases", *PHASES]
if UPDATES: args += ["--updates", str(UPDATES)]
rc = subprocess.call([sys.executable, "scripts/run_plan.py", *args])
print("run_plan exit code", rc)
!python scripts/run_plan.py --status

In [ ]:
# ---- 8. package: results/ as a zip for download; runs/ stay in /kaggle/working for the next session
!du -sh runs results 2>/dev/null
!cd /kaggle/working && zip -qr fireworkers_results.zip wildfire-comm-marl/results && ls -lh fireworkers_results.zip